In [80]:
import os
import pandas as pd

In [102]:
class Link:
    
    def __init__(self, project, material, qty, travel_dist, return_trip_factor, dist_unit, mode, eff):

        self.project = project  # Pass a Project_logestic_manager instance
        self.material = material
        self.qty = qty
        self.travel_dist = travel_dist
        self.return_trip_factor = return_trip_factor
        self.dist_unit = dist_unit
        self.mode = mode
        self.eff = eff
        self.shipping_org = None

    def compute_impact(self):
        if self.shipping_org:
            return self.compute_with_location()
        else:
            dataset = self.project.get_subdataset("Emission")

            if self.mode not in dataset['mode'].tolist():
                
                raise ValueError(f"Mode {self.mode} not found in dataset.")
                
            impact = dataset.loc[dataset['mode'] == self.mode].iloc[[0], 2:] * self.qty * self.return_trip_factor
            impact = impact.iloc[0].to_dict()
             
            return impact  # Assuming impact data starts from 3rd column

    def compute_with_location(self):
        # Dummy implementation for demonstration
        # Actual logic for compute_with_location needs to be added
        return {"GWP": 100, "AP": 50}


In [103]:
class DefaultScenario:
    def __init__(self, name):
        self.name = name
        self.scenario_impact = {}

    def compute_impact(self):
        raise NotImplementedError("compute_impact must be implemented in subclasses")

class Local(DefaultScenario):
    def __init__(self):
        super().__init__("Local")

class Regional(DefaultScenario):
    def __init__(self):
        super().__init__("Regional")

class RegionalC(DefaultScenario):
    def __init__(self):
        super().__init__("Regional_c")

class National(DefaultScenario):
    def __init__(self):
        super().__init__("National")


In [81]:
class ProjectLogisticManager:

    def __init__(self, name, location, data_folder):

        self.name = name
        self.location = location
        self.data_folder = data_folder
        self.links = []
        self.impact = {"GWP": 0.0, "AP": 0.0, "EP": 0.0, "ODP": 0.0, "SFP": 0.0}
        self.subdataset = {}

    def sub_dataset(self):
        for file in os.listdir(self.data_folder):
            file_path = os.path.join(self.data_folder, file)
            if file.endswith(".csv"):
                try:
                    df = pd.read_csv(file_path)
                    dataset_name = os.path.splitext(file)[0]
                    self.subdataset[dataset_name] = df
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")
            elif file.endswith((".xlsx", ".xls")):
                try:
                    df = pd.read_excel(file_path)
                    dataset_name = os.path.splitext(file)[0]
                    self.subdataset[dataset_name] = df
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")

    def get_subdataset(self, sub_dataset):

        if not self.subdataset:
            self.sub_dataset()
        return self.subdataset.get(sub_dataset)

    def create_link(self, material, qty, travel_dist, return_trip_factor, dist_unit, mode, eff):
        
        link = Link(self, material, qty, travel_dist, return_trip_factor, dist_unit, mode, eff)
        self.links.append(link)
        self.impact = self.merge_impacts(self.impact, link.compute_impact())

    @staticmethod
    def merge_impacts(impact1, impact2):

        for key in impact1:
            impact1[key] += impact2.get(key, 0)
        return impact1

    def get_imapct(self):

        return self.impact

In [105]:
data_folder = r"C:\Users\mhtaba\Desktop\pod_lca_git\pod_lca\src\transportation\transportation_dataset"
project = ProjectLogisticManager(name="Building A", location="Seattle", data_folder=data_folder)

In [106]:
project.get_subdataset("Emission")

,mode,Units,GWP,AP,EP,ODP,SFP
0,Truck,tkm,9,3,8,9,8
1,Rail,tkm,3,4,4,4,7
2,Barge,tkm,10,1,4,10,1
3,Air,tkm,4,10,6,5,7


In [107]:
project.create_link ( material="Concrete", qty=500, travel_dist=150, return_trip_factor=1.5, dist_unit="km", mode="Truck",eff=0.9)
project.create_link ( material="Concrete", qty=200, travel_dist=10, return_trip_factor=1.5, dist_unit="km", mode="Rail",eff=0.9)

In [108]:
project.get_imapct()

{'GWP': 7650.0, 'AP': 3450.0, 'EP': 7200.0, 'ODP': 7950.0, 'SFP': 8100.0}

In [3]:
import geopy
from geopy.geocoders import Nominatim

In [4]:
geolocator = Nominatim(user_agent="pod_lca")
location = geolocator.geocode("175 5th Avenue NYC")

In [5]:
print (location.address)

Flatiron Building, 175, 5th Avenue, Manhattan Community Board 5, Manhattan, New York County, City of New York, New York, 10010, United States


In [76]:
from geopy.geocoders import Nominatim
from geopy.geocoders import Photon


geolocator = Photon(user_agent="test")
location = geolocator.geocode("175 5th Avenue NYC")

location.raw

{'geometry': {'coordinates': [-73.98964162240998, 40.741059199999995],
  'type': 'Point'},
 'type': 'Feature',
 'properties': {'osm_id': 264768896,
  'extent': [-73.9898715, 40.7413004, -73.9895014, 40.7407596],
  'country': 'United States',
  'city': 'New York',
  'countrycode': 'US',
  'postcode': '10010',
  'locality': 'Flatiron District',
  'type': 'house',
  'osm_type': 'W',
  'osm_key': 'tourism',
  'housenumber': '175',
  'street': '5th Avenue',
  'district': 'Manhattan',
  'osm_value': 'attraction',
  'name': 'Flatiron Building',
  'state': 'NY'}}

In [79]:
geolocator = Nominatim(user_agent="test")
location = geolocator.geocode("175 5th Avenue NYC")

location.raw

{'place_id': 333036192,
 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
 'osm_type': 'way',
 'osm_id': 264768896,
 'lat': '40.741059199999995',
 'lon': '-73.98964162240998',
 'class': 'tourism',
 'type': 'attraction',
 'place_rank': 30,
 'importance': 9.175936522464359e-05,
 'addresstype': 'tourism',
 'name': 'Flatiron Building',
 'display_name': 'Flatiron Building, 175, 5th Avenue, Manhattan Community Board 5, Manhattan, New York County, City of New York, New York, 10010, United States',
 'boundingbox': ['40.7407596', '40.7413004', '-73.9898715', '-73.9895014']}

In [74]:
location.latitude

47.6038321

In [77]:
from geopy.geocoders import Nominatim, Photon

__author__ = ["POD/LCA Team"]
__copyright__ = "University of Washington"
__license__ = "MIT License"
__email__ = "mhtaba@uw.edu"
__version__ = "0.1.0"


class Location:
    def __init__(self, location):
        self.location = location
        self.geolocator_ph = Photon(user_agent="geoapiExercises")
        self.geolocator_no = Nominatim(user_agent="geoapiExercises")
        self.state = None
        self.city = None
        self.zipcode = None
        self.coords = None
        self.country = None
        self.get_location_info()

    def get_location_info(self):
        try:
            location_data_no = self.geolocator_no.geocode(self.location)
            location_data_ph = self.geolocator_ph.geocode(self.location)

            if location_data_no:
                address = location_data_no.raw.get('address', {})
                self.city = address.get('city') or address.get('town') or address.get('village')
                self.state = address.get('state')
                self.country = address.get('country')
                self.coords = (location_data_no.latitude, location_data_no.longitude)

            if location_data_ph and not self.zipcode:
                properties = location_data_ph.raw.get('properties', {})
                self.zipcode = properties.get('postcode')
        except Exception as e:
            print(f"Error fetching location info: {e}")

    def loc_to_state(self):
        return self.state

    def loc_to_city(self):
        return self.city

    def loc_to_zip(self):
        return self.zipcode

    def loc_to_coordinates(self):
        return self.coords

    def loc_to_egrid(self):
        return self.coords  # TODO: Implement eGRID mapping if dataset is available


if __name__ == '__main__':
    location_input = "175 5th Avenue NYC"
    location_obj = Location(location_input)

    print(f"State: {location_obj.loc_to_state()}")
    print(f"City: {location_obj.loc_to_city()}")
    print(f"Zipcode: {location_obj.loc_to_zip()}")
    print(f"Coordinates: {location_obj.loc_to_coordinates()}")


Error fetching location info: Non-successful status code 403
State: None
City: None
Zipcode: None
Coordinates: None


In [103]:
df = pd.read_csv (r"C:\Users\mhtaba\Desktop\pod_lca_git\pod_lca\data\transportation_dataset\Emission.csv")

In [104]:

df = df[df["CFS_id"] == 4]

In [115]:
df["CFS_id"].values

numpy.ndarray

In [116]:
value = df.loc[0, "CFS_id"]

In [119]:
import pandas as pd

# Example DataFrame
data = {
    'SCTG_code': [1] * 10,
    'distance': [100, 200, 150, 300, 250, 400, 350, 450, 500, 550]
}
df = pd.DataFrame(data)



In [121]:
# Calculate quartiles
quartiles = df['distance'].quantile([0.25, 0.5, 0.75]).values

quartiles

array([212.5, 325. , 437.5])

In [123]:
# Define a function to classify distances into quartiles
def assign_quartile(x, q1, q2, q3):
    if x <= q1:
        return 'Q1'
    elif x <= q2:
        return 'Q2'
    elif x <= q3:
        return 'Q3'
    else:
        return 'Q4'

# Apply the function and create a new column for quartiles
df['quartile'] = df['distance'].apply(assign_quartile, args=(quartiles[0], quartiles[1], quartiles[2]))
df


,SCTG_code,distance,quartile
0,1,100,Q1
1,1,200,Q1
2,1,150,Q1
3,1,300,Q2
4,1,250,Q2
5,1,400,Q3
6,1,350,Q3
7,1,450,Q4
8,1,500,Q4
9,1,550,Q4


In [124]:
# Split into four DataFrames for each quartile
q1_df = df[df['quartile'] == 'Q1']
q2_df = df[df['quartile'] == 'Q2']
q3_df = df[df['quartile'] == 'Q3']
q4_df = df[df['quartile'] == 'Q4']

# Display the results
print("Quartile 1 DataFrame:")
print(q1_df)
print("\nQuartile 2 DataFrame:")
print(q2_df)
print("\nQuartile 3 DataFrame:")
print(q3_df)
print("\nQuartile 4 DataFrame:")
print(q4_df)

Quartile 1 DataFrame:
   SCTG_code  distance quartile
0          1       100       Q1
1          1       200       Q1
2          1       150       Q1

Quartile 2 DataFrame:
   SCTG_code  distance quartile
3          1       300       Q2
4          1       250       Q2

Quartile 3 DataFrame:
   SCTG_code  distance quartile
5          1       400       Q3
6          1       350       Q3

Quartile 4 DataFrame:
   SCTG_code  distance quartile
7          1       450       Q4
8          1       500       Q4
9          1       550       Q4


In [127]:
q4_df["distance"].mean()

500.0